In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw.csv")
print (df.dtypes)

customer_id                   str
gender                        str
age                         int64
country                       str
city                          str
customer_segment              str
tenure_months               int64
signup_channel                str
contract_type                 str
monthly_logins              int64
weekly_active_days          int64
avg_session_time          float64
features_used               int64
usage_growth_rate         float64
last_login_days_ago         int64
monthly_fee                 int64
total_revenue               int64
payment_method                str
payment_failures            int64
discount_applied              str
price_increase_last_3m        str
support_tickets             int64
avg_resolution_time       float64
complaint_type                str
csat_score                float64
escalations                 int64
email_open_rate           float64
marketing_click_rate      float64
nps_score                   int64
survey_respons

In [74]:
contract_type =df['contract_type'].value_counts().reset_index()
print(contract_type)

  contract_type  count
0       Monthly   4967
1     Quarterly   3050
2        Yearly   1983


CLEANING

In [31]:
# Replacement comma with dot
columns_fix = ['avg_session_time',
               'usage_growth_rate',
               'avg_resolution_time',
               'email_open_rate',
               'marketing_click_rate']

for col in columns_fix:
    if df[col].dtype == 'object':
        df[col] = df[col].str.replace(',', '.').astype(float)

print(df[columns_fix].dtypes)

avg_session_time        float64
usage_growth_rate       float64
avg_resolution_time     float64
email_open_rate         float64
marketing_click_rate    float64
dtype: object


In [32]:
print(df['discount_applied'].unique())

<StringArray>
['Yes', 'No']
Length: 2, dtype: str


In [ ]:
# Replace string to Boolean
df['discount_applied'] = df['discount_applied'].map({'Yes':1, 'No':0})
df['price_increase_last_3m'] = df['price_increase_last_3m'].map({'Yes':1, 'No':0})

print(df['discount_applied'].unique())
print(df['price_increase_last_3m'].unique())

[1 0]
[0 1]


In [34]:
# Check missing values
missing_data = df.isnull().sum()
missing_percentage = (df.isnull().sum()/len(df)) * 100

missing_report = pd.DataFrame ({
    'Total Missing': missing_data,
    'Percentage (%)': missing_percentage
}).sort_values(by='Total Missing', ascending=False)

print(missing_report)

                        Total Missing  Percentage (%)
complaint_type                   2045           20.45
customer_id                         0            0.00
age                                 0            0.00
gender                              0            0.00
city                                0            0.00
customer_segment                    0            0.00
tenure_months                       0            0.00
country                             0            0.00
contract_type                       0            0.00
monthly_logins                      0            0.00
weekly_active_days                  0            0.00
avg_session_time                    0            0.00
features_used                       0            0.00
usage_growth_rate                   0            0.00
last_login_days_ago                 0            0.00
signup_channel                      0            0.00
monthly_fee                         0            0.00
total_revenue               

In [48]:
#Flagging strategy
no_complaint_data = df[df['complaint_type'] == 'None']
print(no_complaint_data['support_tickets'].sum())

0


In [46]:
print(
    df.groupby('complaint_type')['support_tickets'].sum()
)

complaint_type
Billing      2869
Service      2520
Technical    4288
Name: support_tickets, dtype: int64


In [ ]:
suspicious_zeros = df[(df['support_tickets'] > 0) & (df['avg_resolution_time'] == 0)]
print(f"Rows with tickets but zero resolution time: {len(suspicious_zeros)}")


Rows with tickets but zero resolution time: 0


In [ ]:
#check for duplicates 
duplicates = df.duplicated(subset=['customer_id']).sum()
print(f'Duplicate Customer IDs: {duplicates}')

Duplicate Customer IDs: 0


In [61]:
# Check for logical Consistency
# total_revenue vs monthly_fee 
# &
# weekly_active_days > 0 vs monthly_logins = 0
    
suspicious_revenue  = df[
    (df['tenure_months'] > 1) &
    (df['total_revenue'] < df['monthly_fee'])
]
print(f"Revenue consistency errors: {len(suspicious_revenue)}")
#revenue validation check ✔

suspicious_login = df[
    (df['weekly_active_days'] > 0) &
    (df['monthly_logins'] == 0)
]
print(f"Usage consistency errors: {len(suspicious_login)}")

Revenue consistency errors: 0
Usage consistency errors: 263


In [63]:
# If someone is active 3 days a week, 
# we consider them to have at least 4 logins per month (1 per week minimum)
df.loc[(df ['weekly_active_days'] > 0) & (df ['monthly_logins'] == 0), 'monthly_logins'] = df['weekly_active_days'] * 4

#Re-check
login_errors_after = df[(df['weekly_active_days'] > 0) & (df['monthly_logins'] == 0)]
print(f"Remaining Usage consistency errors: {len(login_errors_after)}")

Remaining Usage consistency errors: 0


In [75]:
# Save the cleaning data
df.to_csv('../data/saas_churn_cleaned.csv', index=False)
print("Cleaning Phase Completed. Data saved to processed folder.")


Cleaning Phase Completed. Data saved to processed folder.
